# Tokenizer、模型與文字生成：打開 LLM 的黑盒子

<a href="https://colab.research.google.com/github/tjwei/DataScienceSoftware2026/blob/main/06-transformers/02-Tokenizer-Model-Generate.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

建議環境：Google Colab，`執行階段 → 變更執行階段類型 → T4 GPU`

In [ ]:
!pip install -q -U transformers
# 下載本章練習的參考解答 (q_*.py)
import os, urllib.request
BASE_URL = 'https://raw.githubusercontent.com/tjwei/DataScienceSoftware2026/main/06-transformers/'
for fn in ['q_tokens.py', 'q_sampling.py', 'q_perplexity.py']:
    if not os.path.exists(fn):
        urllib.request.urlretrieve(BASE_URL + fn, fn)

In [ ]:
import time
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
device = "cuda" if torch.cuda.is_available() else "cpu"

## Tokenizer

LLM 看到的不是文字，而是 **token** 編號。現代 LLM 大多使用 BPE (Byte Pair Encoding)：常見的字串合併成一個 token。

In [ ]:
model_id = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(model_id)
print("詞彙量:", len(tokenizer))
ids = tokenizer.encode("Hello world! 今天天氣真好。")
ids

In [ ]:
[tokenizer.decode([i]) for i in ids]

In [ ]:
tokenizer.decode(ids)

In [ ]:
# 一次處理多句，padding 成一樣長，回傳 PyTorch tensor
batch = tokenizer(["你好", "Transformers 是什麼？"], padding=True, return_tensors="pt")
batch

## Q
* 比較同樣意思的中文與英文句子，各需要多少個 token？
* 數字 `"1234567"`、emoji、程式碼會被切成什麼樣子？

In [ ]:
# %load q_tokens.py
%run -i q_tokens.py

## 載入模型

* `AutoModelForCausalLM`：「因果語言模型」，也就是預測下一個 token 的模型 (GPT 類)
* T4 不支援 bf16，所以指定 `dtype=torch.float16`
* `device_map="auto"` 會自動把模型放到 GPU 上

In [ ]:
model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float16, device_map="auto")
print(model.config.num_hidden_layers, "layers,", model.config.hidden_size, "hidden size")
print(f"{model.num_parameters()/1e9:.2f}B 參數, 記憶體: {model.get_memory_footprint()/1e9:.2f} GB")

In [ ]:
model

## 預測下一個 token

模型輸出 `logits`，形狀是 `(batch, 序列長度, 詞彙量)`：每個位置對「下一個 token」的預測分數。

In [ ]:
inputs = tokenizer("台灣最高的山是", return_tensors="pt").to(device)
with torch.no_grad():
    out = model(**inputs)
out.logits.shape

In [ ]:
probs = out.logits[0, -1].float().softmax(-1)
top = torch.topk(probs, 10)
for p, i in zip(top.values, top.indices):
    print(f"{tokenizer.decode(i)!r:12} {p.item():.3f}")

### 自己寫生成迴圈 (greedy decoding)

一次選一個機率最高的 token，接到後面，再預測下一個。

In [ ]:
@torch.no_grad()
def greedy(prompt, max_new_tokens=30):
    ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
    for _ in range(max_new_tokens):
        logits = model(ids).logits[:, -1]
        next_id = logits.argmax(-1, keepdim=True)
        ids = torch.cat([ids, next_id], dim=1)
        if next_id.item() == tokenizer.eos_token_id:
            break
    return tokenizer.decode(ids[0])
t0 = time.perf_counter()
print(greedy("台灣最高的山是", 50))
print(f"{time.perf_counter()-t0:.1f}s")

### KV cache

上面每次都把整段文字重新算一次，很浪費。Attention 中之前位置的 key、value 不會改變，可以存起來重複使用，這就是 **KV cache**。

In [ ]:
@torch.no_grad()
def greedy_kv(prompt, max_new_tokens=30):
    ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
    out = model(ids, use_cache=True)
    generated = []
    for _ in range(max_new_tokens):
        next_id = out.logits[:, -1].argmax(-1, keepdim=True)
        generated.append(next_id.item())
        if next_id.item() == tokenizer.eos_token_id:
            break
        out = model(next_id, past_key_values=out.past_key_values, use_cache=True)  # 只要輸入新的 token
    return tokenizer.decode(ids[0].tolist() + generated)
t0 = time.perf_counter()
print(greedy_kv("台灣最高的山是", 50))
print(f"{time.perf_counter()-t0:.1f}s")

In [ ]:
type(model(**inputs, use_cache=True).past_key_values)

### `model.generate`

實際使用時，`generate` 已經幫我們處理好 KV cache、各種取樣方法、停止條件等等。

In [ ]:
inputs = tokenizer("台灣最高的山是", return_tensors="pt").to(device)
out = model.generate(**inputs, max_new_tokens=50, do_sample=False)
tokenizer.decode(out[0])

In [ ]:
# 取樣 (sampling)
out = model.generate(**inputs, max_new_tokens=50, do_sample=True, temperature=0.7, top_p=0.8, top_k=20,
                     num_return_sequences=3)
for o in out:
    print(tokenizer.decode(o, skip_special_tokens=True))
    print("-----")

## Q
自己實作 sampling 版本的生成迴圈（使用 KV cache），支援 `temperature` 和 `top_p` (nucleus sampling)：
* top-p：把機率由大到小排序，只保留累積機率達到 `p` 的最小集合，再從中抽樣

In [ ]:
# %load q_sampling.py
%run -i q_sampling.py

## Chat template

聊天模型 (instruct model) 訓練時，對話被包裝成特定格式。每個模型的格式不同，`apply_chat_template` 會依照模型自帶的模板處理。

In [ ]:
messages = [
    {"role": "system", "content": "你是一個說話簡潔的助教，使用繁體中文。"},
    {"role": "user", "content": "什麼是 Transformer？"},
]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
print(text)

In [ ]:
inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, enable_thinking=False,
                                       return_dict=True, return_tensors="pt").to(device)
out = model.generate(**inputs, max_new_tokens=256, do_sample=True, temperature=0.7, top_p=0.8, top_k=20)
print(tokenizer.decode(out[0, inputs.input_ids.shape[1]:], skip_special_tokens=True))

## 思考模式 (Reasoning)

Qwen3 這類「推理模型」會先在 `<think> ... </think>` 中寫下思考過程，再回答。思考越久，困難的問題 (數學、程式) 通常答得越好。

In [ ]:
from transformers import TextStreamer
messages = [{"role": "user", "content": "一個籠子裡有雞和兔子，共 35 個頭、94 隻腳，雞和兔子各有幾隻？"}]
inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, enable_thinking=True,
                                       return_dict=True, return_tensors="pt").to(device)
streamer = TextStreamer(tokenizer, skip_prompt=True)   # 一邊生成一邊印出來
out = model.generate(**inputs, max_new_tokens=2048, do_sample=True, temperature=0.6, top_p=0.95, top_k=20,
                     streamer=streamer)

In [ ]:
# 把思考內容和最後的答案分開
gen = out[0, inputs.input_ids.shape[1]:].tolist()
think_end = tokenizer.convert_tokens_to_ids("</think>")
k = len(gen) - gen[::-1].index(think_end) if think_end in gen else 0
print("思考了", k, "個 tokens")
print("答案:", tokenizer.decode(gen[k:], skip_special_tokens=True))

## Q：Perplexity

語言模型對一段文字的 loss（平均每個 token 的 negative log-likelihood），取 exp 就是 perplexity。越低代表模型覺得這段文字越「自然」。

提示：`model(input_ids, labels=input_ids).loss`，模型會自動往後移一格計算 cross entropy。

比較這幾句的 perplexity：
* `"今天天氣很好，我們去公園散步吧。"`
* `"今天公園很好，我們去天氣散步吧。"`
* `"天今很好氣，我公散們去步園吧。"`

In [ ]:
# %load q_perplexity.py
%run -i q_perplexity.py